# L04 Assignment — Kaggle-Style Churn Competition
**⏱️ Suggested time: ~60–90 min (self-study)**

> *NorthStar's analytics team has held out a "scoring set" of 2,000 customers — those rows weren't part of L03/L04 training. Your goal: train your best model on the public training data and submit predictions for the scoring set. The model with the highest F1 (at threshold 0.5) wins.*

This assignment has two parts:

1. **Part A — Kaggle-style competition on NorthStar churn.** Three tiers from "beat the L03 baseline" to "open optimisation."
2. **Part B — Independent banking-fraud detection.** Apply the same toolkit to a completely different domain.

**Sample solutions** are at the bottom of the notebook. Attempt each tier before scrolling.

**Time budget:** ~90 minutes — 60 min for the competition, 30 min for banking fraud.

---

## Setup

In [ ]:
# --- Setup: load the toolbox (same libraries used all week) ---
import numpy as np                     # calculator for lists of numbers
import pandas as pd                    # tables of data
import matplotlib.pyplot as plt        # charts
import warnings                        # hide distracting warnings
import time                            # stopwatch for timing tuning runs

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV  # splitting, fair scoring, auto-tuning
from sklearn.compose import ColumnTransformer      # different prep per column type
from sklearn.pipeline import Pipeline              # prep + model assembly line
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # scale numbers; encode categories
from sklearn.impute import SimpleImputer           # fill missing values
from sklearn.linear_model import LogisticRegression             # baseline model
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier  # the tree ensembles
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report  # scorecards

warnings.filterwarnings("ignore")   # keep output clean
print("✅ Setup complete.")

## Build the public/scoring split

The "scoring set" is held out — your goal is to maximise F1 on it. Same data + split as L04, just relabelled to make the competition framing clear.

In [ ]:
# --- Build the competition split: public training data vs held-out scoring data ---
df = pd.read_csv("data/northstar_churn.csv")     # load the customer table
y  = df["churned"]                                # the answer to predict
X  = df.drop(columns=["customer_id", "churned"])  # the clues

# Public training set (80%) — what you can train on
# Scoring set (20%) — held out; your final F1 is computed here
X_public, X_scoring, y_public, y_scoring = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42,  # same split as L04 so scores are comparable
)

print(f"Public training set: {len(X_public):,} customers ({y_public.mean():.1%} churners)")
print(f"Scoring set:         {len(X_scoring):,} customers ({y_scoring.mean():.1%} churners)")
print()
print("→ Train any model you like on X_public/y_public.")
print("→ Predict on X_scoring → compute F1 vs y_scoring.")

---

## 📚 Choose your track

This assignment has **two tracks**. Pick **one** based on your background — you don't need to do both.

| Track | Who it's for | What you'll do |
|---|---|---|
| **🟢 Foundational Track** | Learners new to ML / programming | Tier 1 (one model F1) + Tier 2 (beat F1 ≈ 0.30) |
| **🔵 Advanced Track** | Learners with prior ML background | Tier 3 (open optimisation) + Part B banking-fraud exercises |

If you're unsure, start with the **Foundational Track**. If it feels easy, skip ahead to the **Advanced Track** — both tracks cover the same lesson outcomes; only the scaffolding differs.

---


---

# 🟢 Foundational Track

> *No prior ML background needed. The cells below are scaffolded — read the worked example, then fill in the blanks. Hints are included.*

---


# Part A — Kaggle-Style Churn Competition

The L03 logistic-regression baseline scored ~0.33 F1 on a similar split (at threshold 0.5, with `class_weight='balanced'`). Can you beat that?

---

## 🟢 Tier 1 — Beat the vanilla baseline

**Task:** train ANY model that beats F1 ≈ 0.20 on the scoring set. Use any algorithm from L03 or L04 with default settings. No tuning required.

*Difficulty ladder:* F1 ≈ 0.20 is the level of the simple untuned models in **NB 01**; you saw it reached with almost no effort there.

Scoring is at the default **threshold 0.5** for all tiers. (Reminder from L03/L04: vanilla logistic regression without class-weight rebalancing collapses to ~F1 0.05 at threshold 0.5 on this imbalanced data. Even a basic tree-based model with `class_weight='balanced'` should pass this tier easily.)

In [ ]:
# === Tier 1 — Train ONE model, predict on scoring set, report F1 ===
# This is a fully worked example — run it, read it, then modify it for Tier 2.

# Number columns vs category columns (same as class notebooks)
numeric_features = ["age", "tenure_months", "num_purchases_quarter",
                    "avg_monthly_spend_gbp", "returns_per_purchase",
                    "last_login_days_ago", "avg_review_polarity",
                    "support_tickets_quarter"]
categorical_features = ["region", "subscription_tier"]

# The usual prep recipe: fill gaps, scale numbers, encode categories
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
                      categorical_features),
])

# Your model — start with Random Forest with class_weight='balanced'
my_model = Pipeline([
    ("prep",  preprocessor),                    # step 1: prep the data
    ("model", RandomForestClassifier(
        n_estimators=200,           # 200 trees voting
        min_samples_leaf=5,         # stop leaves memorising individual customers
        class_weight="balanced",    # crucial: churners are rare but important
        n_jobs=-1, random_state=42,
    )),
])

my_model.fit(X_public, y_public)            # train on the public data only
y_pred = my_model.predict(X_scoring)        # predict for the held-out scoring customers
score = f1_score(y_scoring, y_pred)         # grade yourself

print(f"Tier 1 — Scoring F1 (threshold 0.5): {score:.3f}")
if score > 0.20:
    print("✅ You passed Tier 1. Move to Tier 2.")
else:
    print("⚠ Below the bar. Try class_weight='balanced' if you forgot it.")

## 🟡 Tier 2 — Beat F1 ≈ 0.30

**Task:** train a model that beats F1 ≈ 0.30 on the scoring set. This is roughly the level a tuned Random Forest or a default Gradient Boosting reaches — the models you built in **NB 02** and **NB 03**. You can hand-tune or use a small GridSearchCV.

*Fill in the cell below.*

<details><summary>Hint</summary>

Copy the Tier 1 cell, swap `RandomForestClassifier(...)` for `HistGradientBoostingClassifier(learning_rate=0.05, max_iter=200, max_depth=4, class_weight="balanced", random_state=42)`. That alone usually clears 0.30. A worked sample is at the bottom of this notebook.

</details>

In [ ]:
# === Tier 2 — Beat F1 ≈ 0.30 on the scoring set ===
# (your code here)

# Skeleton — replace with your model:
# my_model_v2 = Pipeline([
#     ("prep",  preprocessor),                          # reuse the prep recipe from Tier 1
#     ("model", HistGradientBoostingClassifier(...)),   # try Gradient Boosting — remember class_weight="balanced"
# ])
# my_model_v2.fit(X_public, y_public)                   # train on public data
# y_pred_v2 = my_model_v2.predict(X_scoring)            # predict on scoring set
# score_v2 = f1_score(y_scoring, y_pred_v2)             # grade yourself
# print(f"Tier 2 — Scoring F1: {score_v2:.3f}")

---

# 🔵 Advanced Track

> *For learners with prior ML background. Minimal scaffolding — you decide the approach. You're welcome to peek at the Foundational Track above for reference.*

---


## 🟠 Tier 3 — Open optimisation

**Task:** submit your **single best model** with cross-validated F1 reported on the public training set, and the scoring-set F1 evaluated at threshold 0.5.

**Rules:**
- Tune as aggressively as you want on `X_public` / `y_public`.
- Do NOT use `X_scoring` / `y_scoring` for any decision before your final evaluation.
- Report both CV F1 (on training) and scoring F1 (on held-out).

**Track to beat:** F1 > 0.35 on the scoring set — roughly what the tuned GridSearchCV winner in **NB 04** achieves.

*Fill in the cell below.*

In [ ]:
# === Tier 3 — Open optimisation: your best effort ===
# (your code here)

# Recommended approach:
# 1. Pick your favourite algorithm (HistGradientBoostingClassifier is a strong default).
# 2. Set up a GridSearchCV with a small but smart grid (8-15 combinations).
# 3. Fit on X_public / y_public.
# 4. Predict on X_scoring.
# 5. Report cv_f1 (from grid.best_score_) AND scoring_f1 (from f1_score on X_scoring).

# Part B — Independent banking-fraud exercises

A different domain: a UK bank wants to flag fraudulent credit-card transactions in real time. Same toolkit applies — but the imbalance is much more extreme (~1.5% fraud rate), the cost of false negatives is high, and the cost of false positives is moderate (a customer call).

Three independent exercises. Each comes with explicit tasks and a blank code cell.

In [ ]:
# --- Create the banking-fraud dataset (synthetic but realistic) ---
# You do NOT need to understand every line here — just RUN it. It builds a table of
# 12,000 credit-card transactions where ~4% are fraud. Skim the comments if curious.
def generate_fraud_data(n=12000, seed=2027):
    rng = np.random.default_rng(seed)   # random-number generator with a fixed seed (repeatable)

    # Transaction characteristics, drawn from realistic distributions:
    amount_gbp     = np.round(rng.lognormal(mean=3.5, sigma=1.2, size=n).clip(1, 5000), 2)  # most small, a few large
    hour_of_day    = rng.integers(0, 24, size=n)                       # what time it happened
    is_weekend     = rng.binomial(1, 0.28, size=n)                     # ~28% on weekends
    days_since_last_txn = rng.exponential(scale=1.5, size=n).clip(0, 90).round(1)  # gap since previous transaction
    txns_last_24h  = rng.poisson(lam=2.0, size=n).clip(0, 30)          # burst of recent activity
    distance_from_home_km = rng.exponential(scale=15, size=n).clip(0, 1000).round(1)  # how far from home

    merchant_category = rng.choice(
        ["grocery", "fuel", "restaurant", "online_retail", "travel", "atm", "other"],
        size=n, p=[0.22, 0.10, 0.15, 0.28, 0.08, 0.10, 0.07])          # type of shop
    card_present = rng.choice(["yes", "no"], size=n, p=[0.55, 0.45])   # physical card vs online

    # Some missing values — real data always has gaps
    missing_amt = rng.random(n) < 0.04           # 4% of amounts go missing
    amount_arr = amount_gbp.astype(float)
    amount_arr[missing_amt] = np.nan

    # Fraud target — heavy imbalance (intercept tuned to ~4% fraud rate)
    # Each factor below nudges the fraud risk up, mimicking real fraud patterns:
    logit = (
        -3.4
        + 0.0010 * amount_gbp                            # higher amount → more fraud risk
        + 1.2    * ((hour_of_day < 5) | (hour_of_day > 22)).astype(float)   # off-hours
        + 0.10   * txns_last_24h                          # bursts of activity
        + 0.01   * distance_from_home_km                  # far from home
        + 1.0    * (merchant_category == "online_retail").astype(float)
        + 0.7    * (merchant_category == "atm").astype(float)
        + 0.9    * (card_present == "no").astype(float)
    )
    prob = 1 / (1 + np.exp(-logit))                  # convert risk score to a probability
    is_fraud = (rng.random(n) < prob).astype(int)    # roll the dice: fraud or not

    # Assemble everything into one table
    return pd.DataFrame({
        "transaction_id":         np.arange(500000, 500000 + n),
        "amount_gbp":             amount_arr,
        "hour_of_day":            hour_of_day,
        "is_weekend":             is_weekend,
        "days_since_last_txn":    days_since_last_txn,
        "txns_last_24h":          txns_last_24h,
        "distance_from_home_km":  distance_from_home_km,
        "merchant_category":      merchant_category,
        "card_present":           card_present,
        "is_fraud":               is_fraud,
    })

fraud = generate_fraud_data()   # build the dataset
print(f"Fraud dataset: {len(fraud):,} transactions · fraud rate: {fraud['is_fraud'].mean():.1%}")
fraud.head()                    # peek at the first 5 rows

### Exercise 1 — Train a baseline + a tree ensemble

**Tasks:**
1. Split features (`X_f`) and target (`y_f` = `is_fraud`). Drop `transaction_id`.
2. Train/test split (80/20), stratify on the target.
3. Build a `ColumnTransformer` for numeric + categorical features.
4. Train two models: `LogisticRegression(class_weight='balanced')` and `RandomForestClassifier(class_weight='balanced')`.
5. Report test-set F1 for both.

*Your code:*

<details><summary>Hint</summary>

This is the Tier 1 pattern on a new dataset: build `prep_f` with the fraud column names, wrap each model in a `Pipeline`, `.fit` on the training split, `.predict` on the test split, and compare `f1_score`s. A worked sample is at the bottom of this notebook.

</details>

In [ ]:
# Exercise 1 — baseline + tree ensemble
# (your code here)
# Tip: this mirrors what you did in Tier 1 — new dataset, same recipe.

*Your interpretation: which model wins, and by how much? Is the gap meaningful?*

> (your answer here)

### Exercise 2 — Gradient Boosting + GridSearchCV

**Tasks:**
1. Build a `HistGradientBoostingClassifier` pipeline with `class_weight='balanced'`.
2. Set up a `GridSearchCV` over `learning_rate`, `max_iter`, and `max_depth` (8–12 total combinations).
3. Fit on training data; report best CV F1 + best params.
4. Evaluate on the test set.

*Your code:*

<details><summary>Hint</summary>

Reuse the GridSearchCV pattern from notebook 04: define a `grid` dict with `model__learning_rate`, `model__max_iter`, `model__max_depth`, then `GridSearchCV(gb_f, grid, cv=cv, scoring="f1", n_jobs=-1)`. A worked sample is at the bottom of this notebook.

</details>

In [ ]:
# Exercise 2 — Gradient Boosting tuned
# (your code here)
# Tip: follow the GridSearchCV pattern from notebook 04, cell "Tune Gradient Boosting".

*Your interpretation: how much did tuning improve over defaults? Worth the wall-clock time?*

> (your answer here)

### Exercise 3 — Threshold for a fraud team's daily capacity

**Scenario.** The bank's fraud investigation team can review **50 transactions per day** from this test set (out of ~2,400). They want the 50 highest-risk transactions surfaced.

**Tasks:**
1. Get probabilities from your tuned GB pipeline on the fraud test set.
2. Find the threshold that flags exactly 50 transactions.
3. Compute precision and recall at that threshold.
4. Report: of the 50 flagged, how many are real fraud?

*Your code:*

<details><summary>Hint</summary>

Get probabilities with `search.predict_proba(X_te_f)[:, 1]`. To flag exactly 50, sort the probabilities and use the 50th-highest as the threshold: `np.sort(proba_f)[-50]`. Then compute precision and recall at that threshold. A worked sample is at the bottom of this notebook.

</details>

In [ ]:
# Exercise 3 — capacity-based threshold
# (your code here)
# Tip: sort the predicted probabilities and pick the 50th-highest as your threshold.

*Your recommendation to the fraud team (2 sentences):*

> (your answer here)

## ✅ Submission checklist

- [ ] Tier 1 complete with F1 > 0.20
- [ ] Tier 2 complete with F1 > 0.30
- [ ] Tier 3 complete with both CV F1 and scoring-set F1 reported
- [ ] Exercise 1: LR + RF trained, both F1s reported, interpretation written
- [ ] Exercise 2: GridSearchCV run with best params reported, test F1 reported
- [ ] Exercise 3: 50-transaction threshold found, precision and recall computed, recommendation written

---

# 📚 Sample solutions

*Compare to your work AFTER attempting each.*

## Sample — Tier 2 (Gradient Boosting tuned)

In [ ]:
# === Sample Tier 2 ===
# One worked solution — a tuned-by-hand Gradient Boosting model
gb_pipe = Pipeline([
    ("prep",  preprocessor),                    # same prep recipe
    ("model", HistGradientBoostingClassifier(
        learning_rate=0.05,       # small, stable correction steps
        max_iter=200,             # 200 boosting rounds
        max_depth=4,              # shallow trees — good "weak learners"
        class_weight="balanced",  # handle the churn imbalance
        random_state=42,
    )),
])
gb_pipe.fit(X_public, y_public)             # train on public data only
y_pred_t2 = gb_pipe.predict(X_scoring)      # predict on the scoring set
score_t2 = f1_score(y_scoring, y_pred_t2)   # grade
print(f"Tier 2 sample — Scoring F1: {score_t2:.3f}")

## Sample — Tier 3 (Open optimisation with GridSearchCV)

In [ ]:
# === Sample Tier 3 ===
# One worked solution — let GridSearchCV pick the best GB settings
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # fair 5-fold test

gb_pipe = Pipeline([
    ("prep",  preprocessor),
    ("model", HistGradientBoostingClassifier(class_weight="balanced", random_state=42)),
])
# The settings menu: 3 × 2 × 3 = 18 combinations
grid = {
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_iter":      [200, 400],
    "model__max_depth":     [3, 4, 6],
}

start = time.time()   # stopwatch on
search = GridSearchCV(gb_pipe, grid, cv=cv, scoring="f1", n_jobs=-1)
search.fit(X_public, y_public)     # try every combo, keep the best
elapsed = time.time() - start

cv_f1 = search.best_score_                     # fair score on the public data
y_pred_t3 = search.predict(X_scoring)          # best model predicts the scoring set
scoring_f1 = f1_score(y_scoring, y_pred_t3)    # final competition score

print(f"Elapsed: {elapsed:.1f}s")
print(f"Best params: {search.best_params_}")
print(f"CV F1 (on X_public):    {cv_f1:.3f}")
print(f"Scoring F1 (on X_scoring): {scoring_f1:.3f}")

## Sample — Exercise 1 (LR + RF on fraud data)

In [ ]:
# === Sample Exercise 1 ===
# Split the fraud data into clues (X_f) and answer (y_f), then train/test
y_f = fraud["is_fraud"]                                     # the answer: fraud or not
X_f = fraud.drop(columns=["transaction_id", "is_fraud"])    # the clues (drop ID and answer)
X_tr_f, X_te_f, y_tr_f, y_te_f = train_test_split(
    X_f, y_f, test_size=0.20, stratify=y_f, random_state=42,  # 80/20, keep fraud % equal in both
)

# Which fraud columns are numbers vs categories
numeric_f = ["amount_gbp", "hour_of_day", "is_weekend", "days_since_last_txn",
              "txns_last_24h", "distance_from_home_km"]
categorical_f = ["merchant_category", "card_present"]

# Prep recipe for the fraud data (same pattern as churn)
prep_f = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler())]), numeric_f),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
                      categorical_f),
])

# Two contenders: logistic regression baseline vs random forest
lr_f = Pipeline([("prep", prep_f),
                 ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))])
rf_f = Pipeline([("prep", prep_f),
                 ("model", RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                                   class_weight="balanced", n_jobs=-1, random_state=42))])

lr_f.fit(X_tr_f, y_tr_f); rf_f.fit(X_tr_f, y_tr_f)   # train both

# Compare their F1 on the unseen test transactions
print(f"LogisticRegression  F1: {f1_score(y_te_f, lr_f.predict(X_te_f)):.3f}")
print(f"Random Forest       F1: {f1_score(y_te_f, rf_f.predict(X_te_f)):.3f}")

## Sample — Exercise 2 (Gradient Boosting + GridSearchCV)

In [ ]:
# === Sample Exercise 2 ===
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # fair 5-fold test

# Gradient Boosting pipeline on the fraud data
gb_f = Pipeline([("prep", prep_f),
                 ("model", HistGradientBoostingClassifier(class_weight="balanced", random_state=42))])
# Small settings menu to search
grid = {
    "model__learning_rate": [0.05, 0.1],
    "model__max_iter":      [100, 200],
    "model__max_depth":     [3, 4, 6],
}
# 2×2×3 = 12 combos
search = GridSearchCV(gb_f, grid, cv=cv, scoring="f1", n_jobs=-1)   # try all 12, keep the winner
search.fit(X_tr_f, y_tr_f)
print(f"Best CV F1: {search.best_score_:.3f}")
print(f"Best params: {search.best_params_}")
print(f"Test F1:    {f1_score(y_te_f, search.predict(X_te_f)):.3f}")

## Sample — Exercise 3 (capacity-based threshold)

In [ ]:
# === Sample Exercise 3 ===

# Use the tuned model from Exercise 2 (search.best_estimator_)
proba_f = search.predict_proba(X_te_f)[:, 1]   # fraud probability for every test transaction

# Find threshold that flags exactly 50 transactions
n_to_flag = 50
threshold = np.sort(proba_f)[-n_to_flag]        # the 50th-highest probability becomes the cut-off
y_pred_e3 = (proba_f >= threshold).astype(int)  # flag everything at or above it

# Count how many flags were real fraud
n_flagged = int(y_pred_e3.sum())
real_fraud = int(((y_pred_e3 == 1) & (y_te_f == 1)).sum())
total_fraud = int(y_te_f.sum())

print(f"Threshold:    {threshold:.3f}")
print(f"Flagged:      {n_flagged}")
print(f"Precision:    {precision_score(y_te_f, y_pred_e3):.3f}")
print(f"Recall:       {recall_score(y_te_f, y_pred_e3):.3f}")
print()
print(f"Recommendation:")
print(f"  Of the {n_flagged} flagged transactions, {real_fraud} are real fraud.")
print(f"  That captures {real_fraud / total_fraud:.0%} of all fraud in this test set.")

## What's next

You've now applied the L04 toolkit to two different domains. The pattern is robust:
1. Pipeline preprocessing
2. `class_weight='balanced'` for imbalanced data
3. Tree ensemble (RF or GB) as the model
4. GridSearchCV for hyperparameters
5. Threshold choice based on operational capacity

**Next session → L05 (Unsupervised Learning).** Marcus's new question — *"can we find natural clusters of customer behaviour WITHOUT labels?"* — opens the unsupervised toolkit: k-means, hierarchical clustering, anomaly detection.